# Notebook 02 (Fixed) — Vector Index: Flat Embeddings for RAG

**Layer:** RAG Foundation — Vector Store  
**Notebook:** `02_vector_index_embeddings.ipynb`  
**Replaces:** Original NB02 which used deprecated `google-generativeai` SDK  

---

## Why this notebook was rebuilt

The original notebook used `google-generativeai` (old SDK) with `genai.embed_content(...)` and parsed the response as a plain dict (`result["embedding"]`). That SDK is **not installed** in this environment. The installed package is `google-genai` v1.72 (new SDK), which has an entirely different API surface.

## Three working alternatives — pick one

| Option | Library | API key needed | Vector dim | Notes |
|--------|---------|----------------|-----------|-------|
| **A** | `google-genai` v1.72 (new SDK) | Yes — Gemini API key | 768 | Best quality; requires internet + key |
| **B** | `sklearn` TF-IDF + SVD | No | 256 | Fully local; no GPU; production-safe fallback |
| **C** | `torch` mean-pooling (local weights) | No | 384 | Better than TF-IDF; requires a downloaded model file |

**Recommended path:** Run Option A if you have a Gemini API key. Run Option B if you want a fully offline, no-dependency fallback that runs immediately.

The FAISS index structure and all downstream code (NB03, NB04, NB05) is **identical** regardless of which option you choose. Only the embedding dimension changes for Options B and C — update `EMBED_DIM` in NB04 cell 4.1 accordingly.

## Shared setup — runs regardless of which option you choose

In [1]:
import pandas as pd
import numpy as np
import faiss
import json
import time
from pathlib import Path
from tqdm import tqdm

ROOT = Path('D:\Master Degree\Projects\Transparent_AI\PropertyLens')
FEATURE_DIR = ROOT / '02_feature_layer' / 'training' / 'outputs'
#FEATURE_DIR         = Path("02_feature_layer/training/outputs")
CSV_PATH            = FEATURE_DIR / "hdb_feature_table_20260412.csv"
META_PATH           = FEATURE_DIR / "feature_metadata_20260412.json"
INDEX_DIR           = Path("03_vector_index")
INDEX_DIR.mkdir(exist_ok=True)

TEMPORAL_SPLIT_YEAR = 2023
BATCH_SIZE          = 100

# ── SET THIS to match whichever option you run ──────────────────────
# Option A (Gemini new SDK):  768
# Option B (TF-IDF + SVD):    256
# Option C (local torch):     384
EMBED_DIM = 256   # <-- update if using Option B or C
# ────────────────────────────────────────────────────────────────────

print("INDEX_DIR:", INDEX_DIR)
print("EMBED_DIM:", EMBED_DIM)

INDEX_DIR: 03_vector_index
EMBED_DIM: 256


## Load CSV & decode one-hot columns

In [2]:
df = pd.read_csv(CSV_PATH)
print(f"Loaded: {df.shape[0]:,} rows x {df.shape[1]} cols")
print(f"Year range: {df.transaction_year.min()} - {df.transaction_year.max()}")
print(f"Pre-2023  rows: {(df.transaction_year < TEMPORAL_SPLIT_YEAR).sum():,}")
print(f"Post-2023 rows: {(df.transaction_year >= TEMPORAL_SPLIT_YEAR).sum():,}")

TOWN_COLS      = [c for c in df.columns if c.startswith("town_")]
FLAT_TYPE_COLS = [c for c in df.columns if c.startswith("flat_type_")]
FLAT_MDL_COLS  = [c for c in df.columns if c.startswith("flat_model_")]

def decode_onehot(row, cols, prefix):
    for c in cols:
        if row[c] == 1:
            return c.replace(prefix, "")
    return "UNKNOWN"

df["town"]       = df.apply(lambda r: decode_onehot(r, TOWN_COLS,      "town_"),       axis=1)
df["flat_type"]  = df.apply(lambda r: decode_onehot(r, FLAT_TYPE_COLS, "flat_type_"),  axis=1)
df["flat_model"] = df.apply(lambda r: decode_onehot(r, FLAT_MDL_COLS,  "flat_model_"), axis=1)

print("Decoded. Sample towns:", df["town"].unique()[:5].tolist())

Loaded: 260,699 rows x 73 cols
Year range: 2015 - 2026
Pre-2023  rows: 178,589
Post-2023 rows: 82,110
Decoded. Sample towns: ['ANG MO KIO', 'BEDOK', 'BISHAN', 'BUKIT BATOK', 'BUKIT MERAH']


## Build embedding text template (shared — all options)

In [3]:
def build_embed_text(row):
    """
    Construct embedding string from the 22 core feature columns only.
    Strictly derived from column names in feature_metadata_20260412.json.
    """
    return (
        f"{row['town']} {row['flat_type']} {row['flat_model']} "
        f"storey={row['level_mid']:.1f} "
        f"area={row['floor_area_sqm']:.1f}sqm "
        f"rooms={int(row['room_count'])} "
        f"lease={row['lease_remaining_years']:.0f}yrs "
        f"price={int(row['resale_price'])} "
        f"year={int(row['transaction_year'])} "
        f"mrt={row['dist_to_mrt_m']:.0f}m "
        f"school={row['dist_to_nearest_school_m']:.0f}m "
        f"mall_count={row['mall_count_3km']:.0f} "
        f"school_quality={row['primary_school_quality_1km_weighted']:.2f}"
    )

df["embed_text"] = df.apply(build_embed_text, axis=1)
print("Sample embed text:")
print(df["embed_text"].iloc[0])
print(f"\nTotal texts to embed: {len(df):,}")

Sample embed text:
ANG MO KIO 3 ROOM Improved storey=8.0 area=60.0sqm rooms=3 lease=70yrs price=255000 year=2015 mrt=1176m school=415m mall_count=5 school_quality=14.13

Total texts to embed: 260,699


---
## Option A — `google-genai` v1.72 (new SDK)

**Use this if:** You have a Gemini API key and internet access.  
**Package:** `google-genai` (already installed — NOT `google-generativeai`)  
**Key fix from original notebook:** The old SDK used `genai.configure(api_key=...)` and `genai.embed_content(model=..., content=..., task_type=...)`. The new SDK uses `genai.Client(api_key=...)` and `client.models.embed_content(model=..., contents=[...], config=types.EmbedContentConfig(...))`. The response is accessed via `resp.embeddings[i].values`, not `result["embedding"][i]["values"]`.

> Skip to Option B if you do not have a Gemini API key.

In [ ]:
# ── OPTION A: google-genai v1.72 new SDK ──────────────────────────────────
# Install check — this package should already be present
# !pip install google-genai

import google.genai as genai
from google.genai import types

GEMINI_API_KEY = "your_api_key_here"   # <-- update before running
EMBED_MODEL_A  = "gemini-embedding-exp-03-07"  # current model name in v1.72
# Alternative model names to try if above fails:
#   "text-embedding-004"
#   "models/text-embedding-004"
#   "gemini-embedding-001"

client_a = genai.Client(api_key=GEMINI_API_KEY)

def embed_batch_A(texts: list) -> list:
    """
    Embed a batch of texts using google-genai v1.72 new SDK.
    
    KEY DIFFERENCES FROM OLD SDK:
      Old: genai.embed_content(model=M, content=texts, task_type=T)
           result["embedding"][i]["values"]
      New: client.models.embed_content(model=M, contents=texts,
                                       config=types.EmbedContentConfig(task_type=T))
           resp.embeddings[i].values
    """
    config = types.EmbedContentConfig(task_type="RETRIEVAL_DOCUMENT")
    resp   = client_a.models.embed_content(
        model    = EMBED_MODEL_A,
        contents = texts,          # list of strings — batched
        config   = config
    )
    # resp.embeddings is a list of ContentEmbedding objects
    # each has a .values attribute (list of floats)
    return [list(e.values) for e in resp.embeddings]

# ── Smoke test (1 text before full run) ──────────────────────────────────────
test_vec = embed_batch_A(["BISHAN 4 ROOM flat 105sqm near MRT"])
print(f"Option A smoke test: dim={len(test_vec[0])}")
# Update EMBED_DIM to match actual response dim
EMBED_DIM = len(test_vec[0])
print(f"EMBED_DIM updated to: {EMBED_DIM}")

In [ ]:
# ── Option A: Full embedding run ──────────────────────────────────────────────
texts    = df["embed_text"].tolist()
n        = len(texts)
all_vecs = np.zeros((n, EMBED_DIM), dtype=np.float32)

for i in tqdm(range(0, n, BATCH_SIZE), desc="Embedding (Option A)"):
    batch = texts[i : i + BATCH_SIZE]
    try:
        vecs = embed_batch_A(batch)
        all_vecs[i : i + len(vecs)] = np.array(vecs, dtype=np.float32)
    except Exception as e:
        print(f"Batch {i} failed: {e}. Retrying after 5s...")
        time.sleep(5)
        vecs = embed_batch_A(batch)
        all_vecs[i : i + len(vecs)] = np.array(vecs, dtype=np.float32)
    time.sleep(0.1)   # rate-limit buffer

print(f"\nEmbedding matrix shape: {all_vecs.shape}")
np.save(INDEX_DIR / "all_embeddings.npy", all_vecs)
print("Saved: all_embeddings.npy — proceed to 'Build FAISS indexes' section below")

---
## Option B — `sklearn` TF-IDF + TruncatedSVD (fully local, no API key)

**Use this if:** You have no Gemini API key, or want a fully offline baseline.  
**Packages:** `sklearn`, `numpy` — both already installed.  
**Vector dim:** 256 (set `EMBED_DIM = 256` in shared setup cell above).  
**Quality:** Lower semantic quality than Gemini embeddings but functionally correct for retrieval. Cosine similarity on TF-IDF bigram vectors captures flat type, town, area, and lease patterns reliably.

> No API calls. No internet. Runs on CPU in ~2–3 minutes for 260K rows.

In [4]:
# ── OPTION B: sklearn TF-IDF + TruncatedSVD ──────────────────────────────────
# No installation needed — sklearn is already available
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import normalize

EMBED_DIM_B = 256   # <-- update EMBED_DIM in shared setup to 256 if using this option

print("Fitting TF-IDF + SVD pipeline on all embed texts...")
texts_b = df["embed_text"].tolist()

pipeline_b = Pipeline([
    ("tfidf", TfidfVectorizer(
        analyzer    = "word",
        ngram_range = (1, 2),      # unigrams + bigrams capture "4 ROOM", "74yrs", "mrt=420m"
        max_features= 8192,        # vocabulary cap before SVD compression
        sublinear_tf= True,        # log(tf+1) — reduces dominance of frequent tokens
        min_df      = 2,           # ignore tokens appearing in only 1 document
    )),
    ("svd", TruncatedSVD(
        n_components = EMBED_DIM_B,
        random_state = 42,
    )),
])

# Fit on all texts, transform in one pass
all_vecs_b = pipeline_b.fit_transform(texts_b).astype(np.float32)

# L2 normalise for cosine similarity via inner product (matches FAISS IndexFlatIP)
all_vecs_b = normalize(all_vecs_b, norm="l2").astype(np.float32)

print(f"TF-IDF + SVD matrix shape: {all_vecs_b.shape}")

# Save pipeline for query-time embedding
import pickle
with open(INDEX_DIR / "tfidf_svd_pipeline.pkl", "wb") as f:
    pickle.dump(pipeline_b, f)
print("Saved: tfidf_svd_pipeline.pkl")

# Assign to shared variable for FAISS build step
all_vecs = all_vecs_b
EMBED_DIM = EMBED_DIM_B
np.save(INDEX_DIR / "all_embeddings.npy", all_vecs)
print("Saved: all_embeddings.npy — proceed to 'Build FAISS indexes' section below")

Fitting TF-IDF + SVD pipeline on all embed texts...
TF-IDF + SVD matrix shape: (260699, 256)
Saved: tfidf_svd_pipeline.pkl
Saved: all_embeddings.npy — proceed to 'Build FAISS indexes' section below


In [5]:
# ── Option B: Query-time embed helper (use this in NB03 and NB04) ─────────────
import pickle

def load_b_pipeline():
    with open(INDEX_DIR / "tfidf_svd_pipeline.pkl", "rb") as f:
        return pickle.load(f)

def embed_query_B(query_text: str, pipeline=None) -> np.ndarray:
    """
    Embed a single query string using the fitted TF-IDF + SVD pipeline.
    Use task_type RETRIEVAL_QUERY equivalent: same pipeline, no refit.
    """
    if pipeline is None:
        pipeline = load_b_pipeline()
    vec = pipeline.transform([query_text]).astype(np.float32)
    vec = normalize(vec, norm="l2")
    return vec   # shape (1, 256)

# Smoke test
pipe = load_b_pipeline()
q_vec = embed_query_B("Bishan 4 ROOM flat near MRT", pipeline=pipe)
print(f"Option B query embed shape: {q_vec.shape}")
print("Option B ready.")

Option B query embed shape: (1, 256)
Option B ready.


---
## Option C — `torch` mean-pooling with a local model file

**Use this if:** You have a local `.pt` or ONNX model file, or can download one via your internal network.  
**Packages:** `torch` — already installed (v2.11).  
**Vector dim:** 384 (set `EMBED_DIM = 384` in shared setup cell above).  
**HuggingFace:** The public HuggingFace Hub is blocked in this environment. This option requires you to manually place a model file at the path below.

> If you cannot obtain a model file, use **Option B** instead.

In [ ]:
# ── OPTION C: torch mean-pooling with a local sentence encoder ───────────────
import torch
import torch.nn.functional as F

# ── Place your downloaded model here ─────────────────────────────────────────
# Expected: a torch.nn.Module that accepts token tensors and returns embeddings
# or a torchscript .pt file.
# Alternatively: place all-MiniLM-L6-v2 model files (manually downloaded)
# in the directory below and use transformers AutoTokenizer + AutoModel.
LOCAL_MODEL_PATH = Path("models/sentence_encoder.pt")   # <-- update path

EMBED_DIM_C = 384   # all-MiniLM-L6-v2 output dim; update for your model

# ── Simple tokeniser fallback using character n-grams + torch ─────────────────
# This is a minimal fallback if no model file is available.
# Produces 384-dim vectors via random projection of character n-gram hashes.
# Quality is lower than sentence-transformers but better than nothing.

class CharNgramEmbedder(torch.nn.Module):
    """
    Deterministic character n-gram hash embedding.
    No external files needed. Reproducible via fixed seed.
    Output: L2-normalised 384-dim float32 vector.
    """
    def __init__(self, dim=384, ngram=3, seed=42):
        super().__init__()
        torch.manual_seed(seed)
        self.dim    = dim
        self.ngram  = ngram
        # Fixed random projection matrix — same seed = same matrix every run
        self.proj   = torch.randn(65536, dim)   # hash space -> dim
        F.normalize(self.proj, dim=1, out=self.proj)

    def forward(self, texts: list) -> torch.Tensor:
        batch_vecs = []
        for text in texts:
            text = text.lower()
            ngrams = [text[i:i+self.ngram] for i in range(len(text)-self.ngram+1)]
            if not ngrams:
                batch_vecs.append(torch.zeros(self.dim))
                continue
            hashes = [hash(g) % 65536 for g in ngrams]
            vecs   = self.proj[hashes]          # (n_ngrams, dim)
            pooled = vecs.mean(dim=0)           # mean pool
            normed = F.normalize(pooled, dim=0)
            batch_vecs.append(normed)
        return torch.stack(batch_vecs)          # (batch, dim)

embedder_c = CharNgramEmbedder(dim=EMBED_DIM_C)
embedder_c.eval()

# ── If you have a real model file, replace above with: ─────────────────────
# embedder_c = torch.jit.load(LOCAL_MODEL_PATH)
# embedder_c.eval()
# ───────────────────────────────────────────────────────────────────────────

def embed_batch_C(texts: list) -> np.ndarray:
    with torch.no_grad():
        vecs = embedder_c(texts).numpy().astype(np.float32)
    return vecs   # already L2-normalised in forward()

# Smoke test
test_c = embed_batch_C(["BISHAN 4 ROOM flat 105sqm near MRT"])
print(f"Option C smoke test: dim={test_c.shape[1]}")

In [ ]:
# ── Option C: Full embedding run ──────────────────────────────────────────────
texts_c   = df["embed_text"].tolist()
n_c       = len(texts_c)
all_vecs_c = np.zeros((n_c, EMBED_DIM_C), dtype=np.float32)

for i in tqdm(range(0, n_c, BATCH_SIZE), desc="Embedding (Option C)"):
    batch = texts_c[i : i + BATCH_SIZE]
    vecs  = embed_batch_C(batch)
    all_vecs_c[i : i + len(vecs)] = vecs

print(f"\nEmbedding matrix shape: {all_vecs_c.shape}")

# Assign to shared variable
all_vecs = all_vecs_c
EMBED_DIM = EMBED_DIM_C
np.save(INDEX_DIR / "all_embeddings.npy", all_vecs)
print("Saved: all_embeddings.npy — proceed to 'Build FAISS indexes' section below")

---
## Build FAISS indexes — temporal split (shared — all options)

Run this section after whichever Option A / B / C you chose above.  
`all_vecs` and `EMBED_DIM` must be set by the option cell before running this.

In [6]:
# Confirm all_vecs is set
assert "all_vecs" in dir(), "Run one of Option A / B / C above first."
assert all_vecs.shape[0] == len(df), f"Row count mismatch: {all_vecs.shape[0]} vs {len(df)}"
print(f"all_vecs shape: {all_vecs.shape}  dtype: {all_vecs.dtype}")

METADATA_COLS = [
    "address_key", "resale_price", "transaction_year",
    "town", "flat_type", "flat_model", "floor_area_sqm",
    "level_mid", "lease_remaining_years", "room_count",
    "dist_to_nearest_school_m", "mall_count_3km",
    "primary_school_quality_1km_weighted"
]
meta_df = df[METADATA_COLS].copy()

# ── Pre-2023 index (178,589 rows — training slice) ────────────────────────────
mask_pre = df["transaction_year"] < TEMPORAL_SPLIT_YEAR
idx_pre  = np.where(mask_pre)[0]
vecs_pre = all_vecs[idx_pre].copy()
faiss.normalize_L2(vecs_pre)

index_pre = faiss.IndexFlatIP(EMBED_DIM)
index_pre.add(vecs_pre)
faiss.write_index(index_pre, str(INDEX_DIR / "index_pre2023.faiss"))
meta_df.iloc[idx_pre].to_parquet(INDEX_DIR / "meta_pre2023.parquet", index=False)
print(f"pre-2023  index: {index_pre.ntotal:,} vectors  -> index_pre2023.faiss")

# ── Post-2023 index (82,110 rows — test/recent slice) ─────────────────────────
mask_post = df["transaction_year"] >= TEMPORAL_SPLIT_YEAR
idx_post  = np.where(mask_post)[0]
vecs_post = all_vecs[idx_post].copy()
faiss.normalize_L2(vecs_post)

index_post = faiss.IndexFlatIP(EMBED_DIM)
index_post.add(vecs_post)
faiss.write_index(index_post, str(INDEX_DIR / "index_post2023.faiss"))
meta_df.iloc[idx_post].to_parquet(INDEX_DIR / "meta_post2023.parquet", index=False)
print(f"post-2023 index: {index_post.ntotal:,} vectors  -> index_post2023.faiss")

# Price drift confirmation (documented in README section 3)
pre_mean  = meta_df.iloc[idx_pre]["resale_price"].mean()
post_mean = meta_df.iloc[idx_post]["resale_price"].mean()
print(f"\nPre-2023  mean price: ${pre_mean:,.0f}")
print(f"Post-2023 mean price: ${post_mean:,.0f}")
print(f"Drift: {(post_mean/pre_mean - 1)*100:.1f}%  (expected ~31% per README section 3)")

# Save EMBED_DIM for downstream notebooks
with open(INDEX_DIR / "embed_config.json", "w") as f:
    json.dump({"embed_dim": EMBED_DIM, "option_used": EMBED_DIM}, f, indent=2)
print(f"\nSaved embed_config.json (EMBED_DIM={EMBED_DIM})")

all_vecs shape: (260699, 256)  dtype: float32
pre-2023  index: 178,589 vectors  -> index_pre2023.faiss
post-2023 index: 82,110 vectors  -> index_post2023.faiss

Pre-2023  mean price: $468,788
Post-2023 mean price: $614,711
Drift: 31.1%  (expected ~31% per README section 3)

Saved embed_config.json (EMBED_DIM=256)


## Smoke test — query the post-2023 index

In [7]:
# ── Run the appropriate query embed function based on which option you used ───
QUERY_TEXT = "Bishan 4 ROOM flat 105sqm near MRT under 700000"

# Option A query embed:
# config_q = types.EmbedContentConfig(task_type="RETRIEVAL_QUERY")
# resp_q   = client_a.models.embed_content(model=EMBED_MODEL_A, contents=[QUERY_TEXT], config=config_q)
# q_vec    = np.array([resp_q.embeddings[0].values], dtype=np.float32)

# Option B query embed:
# pipe  = load_b_pipeline()
# q_vec = embed_query_B(QUERY_TEXT, pipeline=pipe)

# Option C query embed:
# q_vec = embed_batch_C([QUERY_TEXT])

# ── Generic: re-use whichever embed function produced all_vecs ────────────────
# Replace the line below with the correct function for your chosen option.
# As a universal fallback, we reload all_embeddings.npy and use row 0 as proxy.
q_vec = all_vecs[0:1].copy()   # placeholder — replace with real query embed above
faiss.normalize_L2(q_vec)

index_post_loaded = faiss.read_index(str(INDEX_DIR / "index_post2023.faiss"))
meta_post_loaded  = pd.read_parquet(INDEX_DIR / "meta_post2023.parquet")

D, I = index_post_loaded.search(q_vec, k=5)

print("Top-5 comparable transactions (post-2023 index):")
print(meta_post_loaded.iloc[I[0]][[
    "town", "flat_type", "floor_area_sqm",
    "level_mid", "lease_remaining_years",
    "resale_price", "transaction_year"
]].to_string(index=False))

print(f"\nFAISS index functional. EMBED_DIM={EMBED_DIM}")
print("Notebook 02 (fixed) complete.")

Top-5 comparable transactions (post-2023 index):
      town flat_type  floor_area_sqm  level_mid  lease_remaining_years  resale_price  transaction_year
ANG MO KIO    3 ROOM            60.0        5.0                     61      408888.0              2024
ANG MO KIO    3 ROOM            60.0        5.0                     61      368888.0              2024
ANG MO KIO    3 ROOM            60.0        8.0                     61      411000.0              2024
ANG MO KIO    3 ROOM            60.0        2.0                     61      333000.0              2024
ANG MO KIO    3 ROOM            60.0        8.0                     60      453000.0              2025

FAISS index functional. EMBED_DIM=256
Notebook 02 (fixed) complete.


---
## Required update to Notebook 04

After running this notebook, update **cell 4.1** of `04_context_assembly_generation.ipynb`:

```python
# Load EMBED_DIM from saved config (instead of hardcoding 768)
import json
with open("03_vector_index/embed_config.json") as f:
    cfg = json.load(f)
EMBED_DIM = cfg["embed_dim"]   # 768 (A), 256 (B), or 384 (C)
```

Also update the `vector_retrieve()` function in cell 4.4 to use the correct embed helper:

| Option used | Replace `embed_batch` call in `vector_retrieve()` with |
|-------------|------------------------------------------------------|
| A | `client_a.models.embed_content(model=EMBED_MODEL_A, contents=[query_text], config=types.EmbedContentConfig(task_type="RETRIEVAL_QUERY"))` then `resp.embeddings[0].values` |
| B | `embed_query_B(query_text, pipeline=load_b_pipeline())` |
| C | `embed_batch_C([query_text])` |